## Imports

In [73]:
# Data access libraries
import ee
import requests
import json
import sys
import pandas
import pip
import platform
from pathlib import Path
from IPython.display import Image, display

# Check for load errors
print(
    "All libraries imported successfully\n"
    f"pip v.{pip.__version__}\n"
    f"Python v{platform.python_version()}"
)

All libraries imported successfully
pip v.25.2
Python v3.13.7


In [57]:
# Verify paths
print(
    f"Working directory: {Path.cwd()}\n" 
    f"Python executable: {sys.executable}\n"
    f"Datasets folder exists: {Path("Datasets").exists()}\n"
    f"Assets folder exists: {Path("Assets").exists()}\n"
)

Working directory: c:\Users\matil\Documents\GitHub\GIS-Project
Python executable: c:\Users\matil\AppData\Local\Programs\Python\Python313\python.exe
Datasets folder exists: True
Assets folder exists: True



## EarthEngine Sentinel-2 & Dynamic World V1


### EE Authentication and AOI


In [59]:
# EE Authentication
ee.Authenticate()
ee.Initialize(project="gis-project-510419")

In [60]:
# Load AOI bounds saved by the DEM notebook
with (Path("Assets") / "dem_AOI.geojson").open(encoding="utf-8") as f:
    gj = json.load(f)

xs, ys = zip(*gj["features"][0]["geometry"]["coordinates"][0])
xmin_geo, ymin_geo, xmax_geo, ymax_geo = min(xs), min(ys), max(xs), max(ys)

print(f"AOI: {xmin_geo, ymin_geo, xmax_geo, ymax_geo}")

AOI: (-123.29161663788408, 49.25498431356825, -122.88478957342635, 49.55071922714218)


In [61]:
# EE AOI, _geo set in DEM bound transforms
aoi = ee.Geometry.Rectangle([
    xmin_geo,
    ymin_geo ,
    xmax_geo,
    ymax_geo
])

### Get Harmonized Sentinel-2 MSI Level-2A (SR)

In [62]:
# Get image from Google Earth Engine
def get_image_collection(year, aoi_bounds, tile, max_cloudy_pixels):
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi_bounds)
        .filterDate(f"{year}-08-01",f"{year}-09-01")
        .filter(ee.Filter.eq('MGRS_TILE', tile))
        .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", max_cloudy_pixels))
    )

# Helper Functions for metadata
def add_band_names(image):
    return image.set("band_names", image.bandNames())

def get_crs(image):
    return image.set("crs", image.select("B4").projection().crs())

# SR Scale 
def scale_reflectance(image):
    scale_factor = 0.0001 # from GEE "The assets contain 12 UINT16 spectral bands representing SR scaled by 10000"
    return image.multiply(scale_factor)

In [63]:
# Image filter parameters
cloudy_pixels_percentage = 10
tile_type = '10UDV'
years = range(2024, 2027)

# Prepare image sets
image_sets = []

for i in years:
    sentinel_set = get_image_collection(i, aoi, tile_type, cloudy_pixels_percentage)
    image_sets.append(sentinel_set)

In [64]:
# Get image metadata from Google Earth Engine
def get_collection_metadata(image_collection):

    metadata =  {
        'No. of Images': image_collection.size().getInfo(),
        'system_index': image_collection.aggregate_array("system:index").getInfo(),
        'system_path': image_collection.aggregate_array('system:id').getInfo(),
        'band_names': image_collection.map(add_band_names).aggregate_array("band_names").getInfo(),
        'date': pd.to_datetime(image_collection.aggregate_array('system:time_start').getInfo(), unit="ms"),
        'crs': image_collection.map(get_crs).aggregate_array("crs").getInfo(),
        
        'PRODUCT_ID': image_collection.aggregate_array('PRODUCT_ID').getInfo(),
        'DATATAKE_IDENTIFIER': image_collection.aggregate_array('DATATAKE_IDENTIFIER').getInfo(),
        'MGRS_TILE': image_collection.aggregate_array('MGRS_TILE').getInfo(),
        'COVERAGE': f"{aoi.area(1).getInfo() / 1e6} km²",

        'CLOUDY_PIXEL_PERCENTAGE': image_collection.aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo(),
        'SENSING_ORBIT_NUMBER': image_collection.aggregate_array('SENSING_ORBIT_NUMBER').getInfo(),
        "MEAN_SOLAR_ZENITH_ANGLE": image_collection.aggregate_array("MEAN_SOLAR_ZENITH_ANGLE").getInfo(),
        "MEAN_SOLAR_AZIMUTH_ANGLE": image_collection.aggregate_array("MEAN_SOLAR_AZIMUTH_ANGLE").getInfo(),
    }

    return metadata

In [65]:
# View tabular image metadata
for image_set in image_sets:
    metadata_sets = get_collection_metadata(image_set)
    display(pd.DataFrame(metadata_sets))

,No. of Images,system_index,system_path,band_names,date,crs,PRODUCT_ID,DATATAKE_IDENTIFIER,MGRS_TILE,COVERAGE,CLOUDY_PIXEL_PERCENTAGE,SENSING_ORBIT_NUMBER,MEAN_SOLAR_ZENITH_ANGLE,MEAN_SOLAR_AZIMUTH_ANGLE
0,4,20240805T191909_20240805T192816_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20240805T191909_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2024-08-05 19:30:57.552,EPSG:32610,S2B_MSIL2A_20240805T191909_N0511_R099_T10UDV_2...,GS2B_20240805T191909_038735_N05.11,10UDV,968.0252025383993 km²,0.493453,99,34.008590,158.502368
1,4,20240807T190911_20240807T191924_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20240807T190911_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2024-08-07 19:20:59.643,EPSG:32610,S2A_MSIL2A_20240807T190911_N0511_R056_T10UDV_2...,GS2A_20240807T190911_047672_N05.11,10UDV,968.0252025383993 km²,0.143168,56,35.177987,154.801963
2,4,20240810T191911_20240810T191959_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20240810T191911_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2024-08-10 19:30:53.911,EPSG:32610,S2A_MSIL2A_20240810T191911_N0511_R099_T10UDV_2...,GS2A_20240810T191911_047715_N05.11,10UDV,968.0252025383993 km²,0.203406,99,35.359129,159.387264
3,4,20240830T191911_20240830T192023_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20240830T191911_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2024-08-30 19:30:53.893,EPSG:32610,S2A_MSIL2A_20240830T191911_N0511_R099_T10UDV_2...,GS2A_20240830T191911_048001_N05.11,10UDV,968.0252025383993 km²,0.002077,99,41.588707,163.519978


,No. of Images,system_index,system_path,band_names,date,crs,PRODUCT_ID,DATATAKE_IDENTIFIER,MGRS_TILE,COVERAGE,CLOUDY_PIXEL_PERCENTAGE,SENSING_ORBIT_NUMBER,MEAN_SOLAR_ZENITH_ANGLE,MEAN_SOLAR_AZIMUTH_ANGLE
0,5,20250802T190931_20250802T191621_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20250802T190931_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2025-08-02 19:21:19.103,EPSG:32610,S2C_MSIL2A_20250802T190931_N0511_R056_T10UDV_2...,GS2C_20250802T190931_004745_N05.11,10UDV,968.0252025383993 km²,2.863577,56,33.831241,153.959003
1,5,20250810T191909_20250810T192442_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20250810T191909_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2025-08-10 19:30:57.008,EPSG:32610,S2B_MSIL2A_20250810T191909_N0511_R099_T10UDV_2...,GS2B_20250810T191909_044026_N05.11,10UDV,968.0252025383993 km²,0.001202,99,35.292058,159.345479
2,5,20250812T190931_20250812T191308_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20250812T190931_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2025-08-12 19:21:17.741,EPSG:32610,S2C_MSIL2A_20250812T190931_N0511_R056_T10UDV_2...,GS2C_20250812T190931_004888_N05.11,10UDV,968.0252025383993 km²,0.003149,56,36.451335,155.887888
3,5,20250822T190931_20250822T191956_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20250822T190931_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2025-08-22 19:21:18.207,EPSG:32610,S2C_MSIL2A_20250822T190931_N0511_R056_T10UDV_2...,GS2C_20250822T190931_005031_N05.11,10UDV,968.0252025383993 km²,0.001735,56,39.417989,158.105614
4,5,20250825T191931_20250825T192858_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20250825T191931_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2025-08-25 19:31:14.376,EPSG:32610,S2C_MSIL2A_20250825T191931_N0511_R099_T10UDV_2...,GS2C_20250825T191931_005074_N05.11,10UDV,968.0252025383993 km²,0.001954,99,39.830207,162.508912


,No. of Images,system_index,system_path,band_names,date,crs,PRODUCT_ID,DATATAKE_IDENTIFIER,MGRS_TILE,COVERAGE,CLOUDY_PIXEL_PERCENTAGE,SENSING_ORBIT_NUMBER,MEAN_SOLAR_ZENITH_ANGLE,MEAN_SOLAR_AZIMUTH_ANGLE
0,6,20260805T191909_20260805T193001_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20260805T191909_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2026-08-05 19:30:54.598,EPSG:32610,S2B_MSIL2A_20260805T191909_N0512_R099_T10UDV_2...,GS2B_20260805T191909_049174_N05.12,10UDV,968.0252025383993 km²,0.000624,99,33.887102,158.400602
1,6,20260807T190911_20260807T191738_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20260807T190911_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2026-08-07 19:20:58.725,EPSG:32610,S2C_MSIL2A_20260807T190911_N0512_R056_T10UDV_2...,GS2C_20260807T190911_010036_N05.12,10UDV,968.0252025383993 km²,0.001403,56,35.053032,154.702427
2,6,20260812T190909_20260812T191707_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20260812T190909_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2026-08-12 19:20:56.841,EPSG:32610,S2B_MSIL2A_20260812T190909_N0512_R056_T10UDV_2...,GS2B_20260812T190909_049274_N05.12,10UDV,968.0252025383993 km²,5.967114,56,36.407918,155.700015
3,6,20260817T190911_20260817T191923_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20260817T190911_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2026-08-17 19:20:56.270,EPSG:32610,S2C_MSIL2A_20260817T190911_N0512_R056_T10UDV_2...,GS2C_20260817T190911_010179_N05.12,10UDV,968.0252025383993 km²,5.302232,56,37.848042,156.778319
4,6,20260820T191911_20260820T192259_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20260820T191911_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2026-08-20 19:30:51.723,EPSG:32610,S2C_MSIL2A_20260820T191911_N0512_R099_T10UDV_2...,GS2C_20260820T191911_010222_N05.12,10UDV,968.0252025383993 km²,0.003795,99,38.178547,161.253204
5,6,20260825T191909_20260825T192922_T10UDV,COPERNICUS/S2_SR_HARMONIZED/20260825T191909_20...,"[B1, B2, B3, B4, B5, B6, B7, B8, B8A, B9, B11,...",2026-08-25 19:30:51.739,EPSG:32610,S2B_MSIL2A_20260825T191909_N0512_R099_T10UDV_2...,GS2B_20260825T191909_049460_N05.12,10UDV,968.0252025383993 km²,2.754107,99,39.770158,162.316087


In [66]:
# Mask bad pixels, keeping 4: vegetation, 5: no vegetation
def SCL_mask(image):
    scl = image.select("SCL")
    good_pixels = scl.gte(4).And(scl.lte(5))
    return image.updateMask(good_pixels)

# Compute median over August images 
def median_composite(images, band_list):
    SCL_masked_images = images.map(SCL_mask)
    band_masked_images = SCL_masked_images.select(band_list)
    scale_masked_images = band_masked_images.map(scale_reflectance)

    source_projection = ee.Image(images.first()).select("B4").projection()

    return (
        scale_masked_images
        .median()
        .setDefaultProjection(source_projection)
    )

# Compute NDVI
def ndvi(images):
    ndvi = images.normalizedDifference(["B8", "B4"])

    return ndvi

In [67]:
# Prepare datasets with median and NDVI
bands = ["B4", "B3", "B2", "B8"]
full_composite_sets = []

for image_set in image_sets:
    composite_set =  median_composite(image_set, bands)
    ndvi_set = ndvi(composite_set)
    full_composite_set = composite_set.addBands(ndvi_set)

    full_composite_sets.append(full_composite_set)

In [68]:
# Preview image of interest (IOI)
try:
    img = full_composite_sets[0] # IOI
    url = img.select(["nd"]).getThumbURL({
        "region": aoi,                   
        "dimensions": 500,           
        "min": -1, 
        "max": 1,           
        "format": "png",
    })

    print(f"Generated URL: {url}\n")
    
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Server Error ({response.status_code}):")
        print(response.text) 
    else:
        print("URL is valid! Attempting display...")
        display(Image(url=url))

except Exception as e:
    print(f"Script execution failed: {e}")

Generated URL: https://earthengine.googleapis.com/v1/projects/gis-project-510419/thumbnails/c3c2fab81ef896cb3b36bee527591a6a-94e4ce8c4f0b9e81218a2fcdc29bc0c5:getPixels

URL is valid! Attempting display...


In [69]:
# Export image as task to Google Drive
def task(export_bands_set, year, aoi_bounds, tile):
    export_task = ee.batch.Export.image.toDrive(
        image=export_bands_set,
        description=f"sentinel_l2a_{year}08_{tile}_composite",
        folder="EarthEngine",
        fileNamePrefix=f"sentinel_l2a_{year}08_{tile}_composite",
        region=aoi_bounds,
        scale=10,
        crs="EPSG:32610",
        maxPixels=1e9,
        fileFormat="GeoTIFF", 
        formatOptions={"noData": -9999},
    )

    export_task.start()
    print("Export started")
    return export_task

# Select export bands
def sentinel_export(sentinel_bands):
    return (
        sentinel_bands
        .select(["B4", "B3", "B2", "B8", "nd"])
        .toFloat()
        .unmask(-9999, sameFootprint=False)
    )

In [ ]:
# Prepare export datasets and start exports
export_bands = []

for full_composite_set in full_composite_sets:
    export_band = sentinel_export(full_composite_set)
    export_bands.append(export_band)
    
tasks = [
    task(export_band, year, aoi, tile_type)
    for year, export_band in zip(years, export_bands)
]

In [ ]:
# Check export status
for year, export_task in zip(years, tasks):
    print(year, export_task.status())

### Get Dynamic World V1

In [70]:
# Get image from Google Earth Engine
def add_coverage(image):
    # Fraction of the AOI where this image has data (1.0 = full coverage)
    cover = (
        image.select("label").mask().unmask(0)
        .reduceRegion(ee.Reducer.mean(), aoi, 100, maxPixels=1e9)
        .get("label")
    )
    return image.set("aoi_coverage", cover)

dw = (
    ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
    .filterDate("2026-08-01", "2026-09-01")
    .filterBounds(aoi)
    .map(add_coverage)
    .filter(ee.Filter.gte("aoi_coverage", 0.95))
)

print( 
    f"Number of images: {dw.size().getInfo()}\n"
    f"Images: {dw.aggregate_array("system:index").getInfo()}\n"
)

Number of images: 2
Images: ['20260807T190911_20260807T191738_T10UDV', '20260807T190911_20260807T191738_T10UEV']



In [71]:
# Image filters
dw_image = dw.filter(
    ee.Filter.eq(
        "system:index",
        "20260807T190911_20260807T191738_T10UDV"
    )
).first()

# Get label metadata
print( 
    f"image ID: {dw_image.getInfo()["id"]}\n"
    f"bands: {dw_image.bandNames().getInfo()}\n"
)

image ID: GOOGLE/DYNAMICWORLD/V1/20260807T190911_20260807T191738_T10UDV
bands: ['water', 'trees', 'grass', 'flooded_vegetation', 'crops', 'shrub_and_scrub', 'built', 'bare', 'snow_and_ice', 'label']



In [ ]:
# Export image as task to Google Drive
dw_label = dw_image.select("label").unmask(255).toUint8()

task_dw = ee.batch.Export.image.toDrive(
    image=dw_label,
    description="dynamic_world_20260807_T10UDV",
    folder="EarthEngine",
    fileNamePrefix="dynamic_world_20260807_T10UDV",
    region=aoi,
    scale=10,
    crs="EPSG:32610",
    maxPixels=1e9,
    fileFormat="GeoTIFF",
    formatOptions={"noData": 255},
)

task_dw.start()
print("Export started")

In [ ]:
print(task_dw.status())